# 01. Annotator and Visuals

This notebook runs an annotator to label tumor vs non-tumor data.

We also visually inspect the dataset.

In [ ]:
from src.data.annotator import ClinicalAnnotator

annotator = ClinicalAnnotator(
    data_path="../data/processed/glioma_dataset.h5",
    algorithm="gmm",       
    pca_variance=0.99,     
    spatial_weight=2.0     
)
annotator.annotate() # This writes 'y_tumor' to the HDF5!

In [ ]:
import h5py
# --- 2. Visual Inspection ---
print("\n🔬 Generating Visual Inspection Maps...")
with h5py.File("../data/processed/glioma_dataset.h5", "r") as f:
    coords = f['coords'][:]
    y_tumor = f['y_tumor'][:]
    
    raw_samples = f['sample_ids'][:]
    if isinstance(raw_samples[0], (bytes, np.bytes_)):
        samples = np.char.decode(raw_samples.astype('S'), 'utf-8')
    else:
        samples = raw_samples.astype(str)

    unique_samples = np.unique(samples)
    
    # Pick 4 random slides to inspect
    np.random.seed(42) 
    test_slides = np.random.choice(unique_samples, 4, replace=False)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    for ax, slide in zip(axes, test_slides):
        mask = (samples == slide)
        c_slide = coords[mask]
        l_slide = y_tumor[mask]
        
        # Plot: Blue=Healthy (0), Red=Tumor (1)
        scatter = ax.scatter(c_slide[:, 0], c_slide[:, 1], c=l_slide, cmap='coolwarm', s=10, alpha=0.8)
        
        # Invert Y axis so it matches standard image/microscope viewing
        ax.invert_yaxis()
        ax.set_aspect('equal')
        ax.set_title(f"{slide}\nTumor: {np.sum(l_slide)} / {len(l_slide)} pixels")
        ax.axis('off')

    plt.tight_layout()
    plt.show()

Initializing Loader...
📂 Loading data from ../data/processed/glioma_dataset.h5...
🔬 'tumor_only=True': Filtering out healthy tissue...
   ✅ Filtered to 189277 Pure Tumor Spectra.
✅ Loader initialized. Total Tumor Spectra: 189277
